# Reconstructed jet-selection influence

Compare inclusive-jet distributions after the standard jet-ID selection (`jetId`), the track-maximum requirement only (`trkMax`), and before either selection (`noSel`). The notebook first displays the stored two-dimensional Gen, Reco, matched-Ref, and RefSel inputs, then projects $\eta^{jet}$ in each configured jet-$p_T$ interval in both the Lab and CM frames. Reco, matched Ref, RefSel, and Gen are overlaid on individual full-distribution canvases, while all ratios to Gen are overlaid on separate canvases. Unit-integral normalization is the default so the figures compare distribution shapes.

Input resolution uses reusable files for the requested jet-selection stem, then the original ROOT file, and raises an error if neither exists. Source eta-cut tables must match the current producer; stale inputs require regeneration. `ETA_CUT_VALUE` (where present) is the physical per-jet acceptance, default 1.9; its index is derived from the shared table. `SINGLE_JET_BINNING` and `DIJET_BINNING` independently select existing `normal` or `test` intervals. Existing `PTAVE_BIN_SET` controls use `standard` as the name for normal binning. Trigger-specific normal bins remain the notebook defaults; test dijet bins use MB 50–100, Jet80 100–180, and Jet100 above 180 GeV. Jet60 has no test interval. Diagnostic frame scans and trigger-stitching overlap windows retain their established definitions.

**Statistical contract.** These are inclusive single-jet eta distributions, not dijet eta or pTave. Rebin first, then normalize in the configured window. Existing B defaults are retained for numerical reproducibility only. B is not validated for Reco/Gen or Ref/Gen (reconstructed-pT migration), nor for independently normalized shapes. RefSel/Gen additionally requires unique truth matching and consistent weights before a raw subset interpretation is justified. The input histograms cannot establish that condition or reconstruct shared-event covariance.

The numerical diagnostics compare B, fixed-normalization independent errors, and delta-method independent-shape errors including random normalization sums. Neither independent approximation measures the missing shared-event correlations. Zero denominators are explicitly marked invalid in diagnostics; ROOT plots retain zero content/error in those bins. No uncertainty is reduced by assuming an unmeasured covariance.


<!-- detailed-workflow-guide -->

### Detailed workflow and normalization

Reco, matched Ref, selected Ref, and inclusive Gen distributions diagnose how reconstruction and jet selections reshape the spectrum. Ratios such as Reco/Gen compare stored weighted yields unless a normalization mode is selected. Unit-area shape normalization uses $\tilde N_i=N_i/\sum_jN_j$; density normalization uses $\tilde N_i=N_i/(\Delta x_i\sum_jN_j)$.

The distinction matters for unequal bin widths. A shape ratio tests relative form after discarding total normalization, whereas a yield ratio retains rate differences. Lab, flipped-Lab, and CM eta coordinates are separate stored observables; the vertical acceptance lines are visual references and do not apply an additional selection.

## Environment and imports

This notebook locates the repository dynamically and imports PyROOT from the
active project environment. Start Jupyter from the repository root with
`py-env/bin/python -m jupyter notebook`; no machine-specific ROOT paths are
added at runtime.


In [ ]:
# Cell role: initialize the reproducible Python/ROOT environment and shared helpers.
# Interpretation: No physics histogram is modified here; ROOT ownership is configured before files open.
# The preceding Markdown gives the equations and physics assumptions for this step.
%load_ext autoreload
%autoreload 2

from dataclasses import replace
from pathlib import Path
import sys

# Locate the repository without relying on a machine-specific absolute path.
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "CMakeLists.txt").is_file()
        and (candidate / "hist_analysis").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Cannot locate the jetAnalysis repository. Start Jupyter from its root."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root

# Batch mode keeps plots reproducible and sends them to notebook/output files.
ROOT = load_root(batch=True)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import SINGLE_JET_PT_BINS
from hist_analysis.python.histogram_io import (
    load_histogram, resolve_combined_file, resolve_direction_file,
)
from hist_analysis.python.jet_selection import (
    normalize_eta_curves, ratio_display_range, selection_ratio_to_gen,
    project_selection_eta, selection_diagnostics,
)
from hist_analysis.python.histogram_ops import ratio_to_nominal
from hist_analysis.python.plotting import draw_overlay
from hist_analysis.python.plot_labels import beam_sample_label, CMS_SIMULATION_HEADER, PPB_HEADER_RIGHT
from hist_analysis.python.root_style import (
    DEFAULT_PLOT_STYLE, configure_cms_header, draw_text_block, save_canvas, set_2d_style,
    set_pad_style, set_palette_style,
)

CMS_HEADER_LEFT = CMS_SIMULATION_HEADER
CMS_HEADER_RIGHT = PPB_HEADER_RIGHT
configure_cms_header(CMS_HEADER_LEFT, CMS_HEADER_RIGHT)

from hist_analysis.python.root_style import configure_mplhep_export

SAVE_MPLHEP = True  # retain ROOT output and optionally add mplhep PDFs
SAVE_MPLHEP_PNG = False
configure_mplhep_export(SAVE_MPLHEP, save_png=SAVE_MPLHEP_PNG)


In [ ]:
# Cell role: perform analysis step 2.
# Interpretation: Operations use the binning, normalization, and uncertainty conventions documented above.
# The preceding Markdown gives the equations and physics assumptions for this step.
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)
ROOT.TH1.AddDirectory(False)

## Configuration

The defaults use the combined embedding file produced with the `jetId` file stem. `PT_BINS` controls the half-open projection intervals $[p_{T}^{min},p_{T}^{max})$. Each Lab and CM comparison also writes a standalone RefSel / Gen figure. Set `REFSEL_GEN_YLIM` to `None` for automatic y-axis scaling. The histogram keys and style indices below are analysis inputs and are kept explicit for auditability.

In [ ]:
SINGLE_JET_BINNING = 'normal'  # normal or test; independent of dijet binning
from hist_analysis.config.histograms import pt_bins, eta_cut_index
# Cell role: define and validate user-facing analysis configuration.
# Interpretation: Changing these values can change inputs, selections, binning, normalization, or outputs.
# The preceding Markdown gives the equations and physics assumptions for this step.
GENERATOR = 'embedding'       # embedding or pythia
DIRECTION = 'combined'        # pgoing, Pbgoing, or combined
FILE_STEM = 'jetId'
PT_BINS = tuple(pt_bins('single', SINGLE_JET_BINNING))
REBIN_ETA = 1                 # ROOT TH1.Rebin grouping; must divide the eta-bin count
if isinstance(REBIN_ETA, bool) or not isinstance(REBIN_ETA, int) or REBIN_ETA < 1:
    raise ValueError('REBIN_ETA must be a positive integer')
NORMALIZATION = 'integral'     # none, integral, or bin_width
APPLY_RATIO_SHAPE_SCALING = True  # False: divide raw rebinned yields only
# NORMALIZATION = 'integral'     # none, integral, or bin_width
# Producer convention: etaCM = oriented etaLab - 0.465.
ETA_SHIFT = 0.465
JETID_VALIDITY_ETA_RANGES = {
    'Lab': (-3.6, 3.6),
    'CM': (-3.0, 3.0),
    # 'CM': (-0.8 - ETA_SHIFT, 0.8 - ETA_SHIFT),
}
# Common numerator/Gen window, inside validity and aligned after rebinning.
NORMALIZATION_ETA_RANGES = {'Lab': (-0.8, 0.8), 'CM': (-1.0, 0.3)}
RECO_GEN_RATIO_OPTION = 'B'    # '' independent; 'B' legacy diagnostic, not validated here
REF_GEN_RATIO_OPTION = 'B'     # independently configurable
RATIO_OPTIONS = {'Reco': RECO_GEN_RATIO_OPTION, 'Ref': REF_GEN_RATIO_OPTION, 'RefSel': 'B'}
if any(option not in {'', 'B'} for option in RATIO_OPTIONS.values()):
    raise ValueError("Ratio options must be '' or 'B'")
RATIO_RANGE = (0.8, 1.2)
REFSEL_GEN_YLIM = (0.95, 1.05)  # None uses automatic scaling for standalone RefSel / Gen
TRKMAX_REFSEL_GEN_CM_NORMALIZATION_RANGE = (2.2, 3.4)  # standalone CM ratio only
LOG_Y = False                  # logarithmic y axis for full distributions
ETA_RANGE = (-3.6, 3.6)              # e.g. (-3.0, 3.0), or None for the full axis
DRAW_GRID = True
SAVE_PNG = False
OUTPUT_DIR = PROJECT_ROOT / 'hist_analysis' / 'output' / 'jet_selection'

FRAMES = ('Lab', 'CM')
HISTOGRAM_KEYS = {
    'Lab': {
        'Gen': 'hGenInclusiveJetPtEtaLab',
        'Reco': {
            'jetId': 'hRecoInclusiveJetPtEtaLab',
            'trkMax': 'hRecoInclusiveJetTrkMaxPtEtaLab',
            'noSel': 'hRecoInclusiveJetNoSelPtEtaLab',
        },
        'Ref': {
            'jetId': 'hRefInclusiveJetPtEtaLab',
            'trkMax': 'hRefInclusiveJetTrkMaxPtEtaLab',
            'noSel': 'hRefInclusiveJetNoSelPtEtaLab',
        },
        'RefSel': {
            'jetId': 'hRefSelInclusiveJetPtEtaLab',
            'trkMax': 'hRefSelInclusiveJetTrkMaxPtEtaLab',
            'noSel': 'hRefSelInclusiveJetNoSelPtEtaLab',
        },
    },
    'CM': {
        'Gen': 'hGenInclusiveJetPtEtaCM',
        'Reco': {
            'jetId': 'hRecoInclusiveJetPtEtaCM',
            'trkMax': 'hRecoInclusiveJetTrkMaxPtEtaCM',
            'noSel': 'hRecoInclusiveJetNoSelPtEtaCM',
        },
        'Ref': {
            'jetId': 'hRefInclusiveJetPtEtaCM',
            'trkMax': 'hRefInclusiveJetTrkMaxPtEtaCM',
            'noSel': 'hRefInclusiveJetNoSelPtEtaCM',
        },
        'RefSel': {
            'jetId': 'hRefSelInclusiveJetPtEtaCM',
            'trkMax': 'hRefSelInclusiveJetTrkMaxPtEtaCM',
            'noSel': 'hRefSelInclusiveJetNoSelPtEtaCM',
        },
    },
}
STYLE_INDICES = {'Reco': 0, 'Ref': 1, 'RefSel': 3, 'Gen': 2}
PLOT_STYLE = replace(
    DEFAULT_PLOT_STYLE, annotation_text_size=0.025, legend_text_size=0.025,
    annotation_line_spacing=0.037,
)

def mc_file(generator, direction):
    if direction == 'combined':
        return resolve_combined_file(BASE_DIR, generator, FILE_STEM)
    return resolve_direction_file(BASE_DIR, generator, direction, FILE_STEM)

INPUT_FILE = mc_file(GENERATOR, DIRECTION)
if not INPUT_FILE.exists():
    raise FileNotFoundError(f'Missing configured ROOT file: {INPUT_FILE}')
INPUT_FILE
SINGLE_JET_PT_BINS = pt_bins('single', SINGLE_JET_BINNING)

# Output identity uses the actual configured selections. Each execution gets
# dated renderer directories and unique filenames; earlier files are preserved.
from hist_analysis.python.output_naming import OutputDirectory, OutputSelection, nominal_jet_eta_cut
OUTPUT_SELECTIONS = OutputSelection(tuple(dict.fromkeys(method for frame in FRAMES for keys in HISTOGRAM_KEYS[frame].values() if isinstance(keys, dict) for method in keys)), {'View': ETA_RANGE or (), **{f'Norm{frame}': bounds or () for frame, bounds in NORMALIZATION_ETA_RANGES.items()}, **{f'Valid{frame}': bounds or () for frame, bounds in JETID_VALIDITY_ETA_RANGES.items()}}, PT_BINS, 'jetPt')
OUTPUT_DIR = OutputDirectory(OUTPUT_DIR, OUTPUT_SELECTIONS, directory_selection=FILE_STEM)

# One deterministic ROOT archive; rerunning configuration starts a fresh file.
from hist_analysis.python.notebook_output import NotebookRootOutput
ROOT_OUTPUT = NotebookRootOutput(OUTPUT_DIR, '02_mc_jet_selection')

import warnings
if 'B' in RATIO_OPTIONS.values():
    warnings.warn('Legacy B errors are retained for reproducibility; migration and '
                  'independent shape normalization invalidate a binomial interpretation. '
                  'See the statistical contract and numerical diagnostics.', RuntimeWarning)


## Load and validate inputs

All objects must be two-dimensional histograms. Detached clones are retained after the input ROOT file is closed.

In [ ]:
# Cell role: resolve input files and load the named ROOT objects.
# Interpretation: Loaded objects that outlive their file must be cloned and detached from ROOT directories.
# The preceding Markdown gives the equations and physics assumptions for this step.
histograms_2d = {}
for frame in FRAMES:
    frame_keys = HISTOGRAM_KEYS[frame]
    histograms_2d[(frame, 'Gen')] = load_histogram(
        str(INPUT_FILE), frame_keys['Gen'],
    )
    for level in ('Reco', 'Ref', 'RefSel'):
        for selection, key in frame_keys[level].items():
            histograms_2d[(frame, f'{level} {selection}')] = load_histogram(
                str(INPUT_FILE), key,
            )
not_th2 = [label for label, histogram in histograms_2d.items()
           if not histogram.InheritsFrom('TH2')]
if not_th2:
    raise TypeError(f'Expected TH2 inputs for: {not_th2}')
{label: histogram.ClassName() for label, histogram in histograms_2d.items()}

## Two-dimensional inputs before projection

The Bird palette, logarithmic z scale, shared 2D axis styling, and palette placement follow `02_mc_event_histograms.ipynb`. Each map is shown before any $p_T$ range is applied.

In [ ]:
# Cell role: draw the configured diagnostic figures and retain ROOT objects for display.
# Interpretation: Axis limits and logarithmic scales affect presentation only, not stored bin contents.
# The preceding Markdown gives the equations and physics assumptions for this step.
def draw_jet_map(histogram, frame, label, output_tag):
    plotted = histogram.Clone(f'{histogram.GetName()}_{output_tag}_2d')
    plotted.SetDirectory(0)
    plotted.SetTitle('')
    plotted.GetXaxis().SetTitle('p_{T}^{jet} (GeV)')
    eta_subscript = 'lab' if frame == 'Lab' else 'CM'
    plotted.GetYaxis().SetTitle(f'#eta_{{{eta_subscript}}}^{{jet}}')
    set_2d_style(plotted)

    canvas = ROOT.TCanvas(
        f'c_{output_tag}_2d', '',
        DEFAULT_PLOT_STYLE.canvas_width, DEFAULT_PLOT_STYLE.canvas_height,
    )
    set_pad_style(canvas, grid_x=DRAW_GRID, grid_y=DRAW_GRID)
    canvas.SetRightMargin(DEFAULT_PLOT_STYLE.palette_right_margin)
    plotted.Draw('COLZ1')
    annotations = draw_text_block(canvas, (
        beam_sample_label(GENERATOR.capitalize(), DIRECTION), label, f'{frame} frame',
    ), style=PLOT_STYLE)
    canvas.Modified()
    canvas.Update()
    palette = set_palette_style(plotted)
    canvas.Modified()
    canvas.Update()
    save_canvas(
        canvas, OUTPUT_DIR / f'{output_tag}_2d.pdf', save_png=SAVE_PNG,
    )
    canvas._jet_map_objects = [plotted, palette, *annotations]
    return canvas

map_canvases = {}
for (frame, label), histogram in histograms_2d.items():
    output_tag = (
        f'{GENERATOR}_{DIRECTION}_{frame.lower()}_'
        f'{label.lower().replace(" ", "_")}'
    )
    canvas = draw_jet_map(histogram, frame, label, output_tag)
    map_canvases[(frame, label)] = canvas
    display(canvas)

Each curve and Gen use the same per-frame `NORMALIZATION_ETA_RANGES[frame]` after rebinning. Lab uses [-0.8, 0.8); CM uses [-1.0, 0.3), complete bins in the overlap of the former central window with the valid CM range. The producer uses etaCM = oriented etaLab - 0.465, mapping the user-specified Lab validity [-0.8, 0.8) to CM [-1.265, 0.335). These are bounds in the plotted coordinate; detector-level migration is not corrected by this display restriction. Both boundaries must be exact rebinned edges. Integral mode divides by the yield in that window; bin_width mode additionally divides by eta-bin width. None uses stored yields. Flow bins never enter normalization. Ratios of normalized curves are central-normalized shape ratios, not absolute efficiencies.

`JETID_VALIDITY_ETA_RANGES` restricts jetId ratio displays separately in Lab and CM. Only complete bins inside the validity bounds are displayed; validity bounds themselves need not coincide with bin edges. The exact normalization window must lie inside those bounds. None explicitly disables masking pending a validated calibration boundary; the central normalization window is not a calibration boundary. Full-distribution overlays and retained raw histograms remain available outside the ratio display range. NoSel/trkMax displays are unaffected.

`comparison_results[...]['raw_refsel_to_gen']` retains an unnormalized RefSel/Gen histogram with B errors, independently of display normalization, and is included in the final ROOT archive. RefSel/Gen always uses B; Reco/Gen and Ref/Gen retain independently configurable options. For B, the rebinned numerator and Gen are each normalized before ROOT division with B. There is no subsequent ratio-level scaling. Normalization-integral uncertainty and covariance remain omitted.


`APPLY_RATIO_SHAPE_SCALING=True` normalizes each rebinned histogram before division. Set it to `False` to divide raw rebinned yields directly. This switch affects Reco/Gen, Ref/Gen and RefSel/Gen, preserving their selected error options; distribution-overlay normalization remains controlled by `NORMALIZATION`.

The standalone CM trkMax RefSel / Gen figure always normalizes RefSel and Gen separately to unit integral in `TRKMAX_REFSEL_GEN_CM_NORMALIZATION_RANGE` before division, using the same ROOT error option. The ratio of the window integrals is one.


In [ ]:
# Cell role: project multidimensional inputs into the configured analysis bins.
# Interpretation: Selections are applied before arithmetic so migrations and boundary bins remain explicit.
# The preceding Markdown gives the equations and physics assumptions for this step.
def eta_projection(histogram, frame, label, selection, pt_range):
    low, high = pt_range
    return project_selection_eta(
        histogram, pt_range, rebin=REBIN_ETA,
        name=f'h{label}{selection}Eta{frame}_{low:g}_{high:g}'.replace('.', 'p'),
    )

def add_cm_eta_boundaries(canvas, output=None):
    canvas.cd()
    histogram = next(obj for obj in canvas.GetListOfPrimitives()
                      if obj.InheritsFrom('TH1'))
    y_low, y_high = histogram.GetMinimum(), histogram.GetMaximum()
    lines = []
    for eta, style in ((2.4, 1), (1.9, 2)):
        for sign in (-1, 1):
            line = ROOT.TLine(sign * eta, y_low, sign * eta, y_high)
            line.SetLineColor(ROOT.kBlack)
            line.SetLineStyle(style)
            line.SetLineWidth(2)
            line.Draw('SAME')
            lines.append(line)
    legend = next((obj for obj in canvas.GetListOfPrimitives()
                   if obj.InheritsFrom('TLegend')), None)
    if legend is not None:
        legend.AddEntry(lines[0], '|#eta_{CM}| = 2.4', 'l')
        legend.AddEntry(lines[2], '|#eta_{CM}| = 1.9', 'l')
        legend.Draw()
    canvas._cm_eta_boundary_objects = [*lines, legend]
    canvas.Modified()
    canvas.Update()
    if output is not None:
        save_canvas(canvas, output, save_png=SAVE_PNG)

comparison_results = {}
projection_specs = (
    (frame, pt_range, selection)
    for frame in FRAMES
    for pt_range in PT_BINS
    for selection in HISTOGRAM_KEYS[frame]['Reco']
)
for frame, pt_range, selection in projection_specs:
    eta_subscript = 'lab' if frame == 'Lab' else 'CM'
    eta_title = f'#eta_{{{eta_subscript}}}^{{jet}}'
    low, high = pt_range
    pt_tag = f'{low:g}_{high:g}'.replace('.', 'p')
    raw_curves = {
        'Reco': eta_projection(
            histograms_2d[(frame, f'Reco {selection}')],
            frame, 'Reco', selection, pt_range,
        ),
        'Ref': eta_projection(
            histograms_2d[(frame, f'Ref {selection}')],
            frame, 'Ref', selection, pt_range,
        ),
        'RefSel': eta_projection(
            histograms_2d[(frame, f'RefSel {selection}')],
            frame, 'RefSel', selection, pt_range,
        ),
        'Gen': eta_projection(
            histograms_2d[(frame, 'Gen')],
            frame, 'Gen', selection, pt_range,
        ),
    }
    normalization_range = NORMALIZATION_ETA_RANGES[frame]
    curves = normalize_eta_curves(raw_curves, NORMALIZATION, normalization_range)
    validity_range = JETID_VALIDITY_ETA_RANGES[frame] if selection == 'jetId' else None
    ratio_eta_range = ratio_display_range(
        raw_curves['Gen'], ETA_RANGE, validity_range,
        normalization_range if NORMALIZATION != 'none' else None,
    )
    raw_refsel_ratio = ratio_to_nominal(
        raw_curves['RefSel'], raw_curves['Gen'],
        name=f'raw_refsel_gen_{frame}_{selection}_{pt_tag}', option='B',
    )
    annotations = (
        beam_sample_label(GENERATOR.capitalize(), DIRECTION), f'{frame} frame',
        f'{low:g} < p_{{T}}^{{jet}} < {high:g} GeV',
        selection,
        *(('Normalization: ' + str(normalization_range),) if NORMALIZATION != 'none' else ()),
    )
    output_tag = (
        f'{GENERATOR}_{DIRECTION}_{frame.lower()}_{selection}_'
        f'reco_ref_refsel_to_gen_pt_{pt_tag}'
    )
    y_title = {
        'none': f'dN/d{eta_title}',
        'integral': f'1/N dN/d{eta_title}',
        'bin_width': f'1/N dN/d{eta_title}',
    }[NORMALIZATION]
    canvas = draw_overlay(
        curves, title='',
        x_title=eta_title, y_title=y_title,
        log_y=LOG_Y,
        x_range=ETA_RANGE, annotations=annotations, grid=DRAW_GRID,
        legend_bounds=(0.45, 0.18, 0.70, 0.30),
        output=OUTPUT_DIR / f'{output_tag}.pdf', save_png=SAVE_PNG,
        style=PLOT_STYLE,
    )
    if frame == 'CM':
        add_cm_eta_boundaries(
            canvas, OUTPUT_DIR / f'{output_tag}.pdf',
        )
    # Draw all ratios together on one individual canvas, rather than in a bottom pad.
    ratios = {}
    for label, histogram in curves.items():
        if label == 'Gen':
            continue
        ratio = selection_ratio_to_gen(
            raw_curves[label], raw_curves['Gen'],
            NORMALIZATION if APPLY_RATIO_SHAPE_SCALING else 'none', normalization_range,
            name=f'{output_tag}_{label}_ratio',
            option='B' if label == 'RefSel' else RATIO_OPTIONS[label],
        )
        ratios[label] = ratio
    ratio_annotations = annotations if APPLY_RATIO_SHAPE_SCALING else tuple(
        text for text in annotations if not text.startswith('Normalization:')
    )
    ratio_canvas = draw_overlay(
        {label: ratio for label, ratio in ratios.items()},
        title='', x_title=eta_title, y_title='Ratio to Gen' if NORMALIZATION == 'none' else 'Ratio to Gen',
        x_range=ratio_eta_range, y_range=RATIO_RANGE, reference_y=1.0,
        grid=DRAW_GRID, headroom=1.1, style_indices=STYLE_INDICES,
        annotations=ratio_annotations,
        legend_bounds=(0.45, 0.18, 0.70, 0.30),
        output=OUTPUT_DIR / f'{output_tag}_ratios.pdf', save_png=SAVE_PNG,
        canvas_name=f'{output_tag}_ratios',
        style=PLOT_STYLE,
    )
    if frame == 'CM':
        add_cm_eta_boundaries(
            ratio_canvas, OUTPUT_DIR / f'{output_tag}_ratios.pdf',
        )
    display(ratio_canvas)

    refsel_gen_tag = f'{GENERATOR}_{DIRECTION}_{frame.lower()}_{selection}_refsel_gen_pt_{pt_tag}'
    refsel_gen_annotations = ratio_annotations
    if frame == 'CM' and selection == 'trkMax':
        window = TRKMAX_REFSEL_GEN_CM_NORMALIZATION_RANGE
        refsel_gen = selection_ratio_to_gen(
            raw_curves['RefSel'], raw_curves['Gen'], 'integral', window,
            name=refsel_gen_tag, option=RATIO_OPTIONS['RefSel'],
        )
        refsel_gen_annotations = (
            *(text for text in ratio_annotations if not text.startswith('Normalization:')),
            # f'Normalization: {window}',
        )
    else:
        refsel_gen = ratios['RefSel'].Clone(refsel_gen_tag)
    refsel_gen.SetDirectory(0)
    refsel_gen.SetMinimum()
    refsel_gen.SetMaximum()
    refsel_gen_label = 'RefSel / Gen'
    refsel_gen_canvas = draw_overlay(
        {refsel_gen_label: refsel_gen},
        title='', x_title=eta_title, y_title=refsel_gen_label,
        x_range=ratio_eta_range, y_range=REFSEL_GEN_YLIM, reference_y=1.0,
        grid=DRAW_GRID, headroom=1.1, annotations=refsel_gen_annotations,
        legend_bounds=(0.45, 0.18, 0.70, 0.30 if frame == 'CM' else 0.24),
        style_indices={refsel_gen_label: STYLE_INDICES['RefSel']},
        canvas_name=refsel_gen_tag,
        style=replace(PLOT_STYLE, annotation_x=0.40) if frame == 'CM' else PLOT_STYLE,
    )
    if frame == 'CM':
        add_cm_eta_boundaries(refsel_gen_canvas)
    save_canvas(
        refsel_gen_canvas, OUTPUT_DIR / f'{refsel_gen_tag}.pdf', save_png=SAVE_PNG,
        mplhep_style_indices={refsel_gen_label: tuple(ratios).index('RefSel')},
        mplhep_reference_y=1.0,
    )
    display(refsel_gen_canvas)
    comparison_results[(frame, pt_range, selection)] = {
        'canvas': canvas, 'ratio_canvas': ratio_canvas, 'raw_histograms': raw_curves,
        'refsel_gen_canvas': refsel_gen_canvas,
        'histograms': curves, 'ratios_to_gen': ratios,
        'raw_refsel_to_gen': raw_refsel_ratio, 'ratio_eta_range': ratio_eta_range,
        'statistics': selection_diagnostics(
            raw_curves, NORMALIZATION if APPLY_RATIO_SHAPE_SCALING else 'none',
            normalization_range),
    }
    display(canvas)

## Numerical summary

Print the raw in-range weighted yields before display normalization. These values provide a compact check of the input populations; they are not efficiencies unless the compared histograms share the intended denominator population.

In [ ]:
# Cell role: perform analysis step 7.
# Interpretation: Operations use the binning, normalization, and uncertainty conventions documented above.
# The preceding Markdown gives the equations and physics assumptions for this step.
for (frame, pt_range, selection), result in comparison_results.items():
    yields = {
        label: histogram.Integral()
        for label, histogram in result['raw_histograms'].items()
    }
    print(
        f'{frame:3s} | {pt_range[0]:g}-{pt_range[1]:g} GeV | {selection:6s} '
        f'| Reco={yields["Reco"]:.6g} | Ref={yields["Ref"]:.6g} '
        f'| RefSel={yields["RefSel"]:.6g} '
        f'| Gen={yields["Gen"]:.6g}'
    )
# Persist identical-input comparisons for every eta bin, including empty bins.
import csv
STATISTICS_CSV = OUTPUT_DIR / 'jet_selection_statistics.csv'
statistics_rows = [dict(frame=frame, pt_low=pt[0], pt_high=pt[1], selection=selection, **row)
                   for (frame, pt, selection), result in comparison_results.items()
                   for row in result['statistics']]
with STATISTICS_CSV.open('w', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(statistics_rows[0]))
    writer.writeheader()
    writer.writerows(statistics_rows)
print('Binwise statistical diagnostics:', STATISTICS_CSV)


### ROOT histogram archive
The file below includes plot-time histogram snapshots and retained results, with configuration metadata. Rerunning the configuration cell recreates the same file; rerunning plot cells replaces their keys.

In [ ]:
# Save retained numerical products in addition to exact plot-time snapshots.
NOTEBOOK_ROOT_FILE = ROOT_OUTPUT.finish(globals())
